<a href="https://colab.research.google.com/github/hombremono873/Cuadernos_colab/blob/main/plantilla_RandomForest_AG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1.0 Carga de librerías**

In [ ]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 42.4 MB/s eta 0:00:00


In [ ]:
# Librerías generales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
# Algoritmo Genético - DEAP
from deap import base
from deap import creator
from deap import tools
from deap import algorithms

# Generación de valores aleatorios
import random

In [ ]:
# Download latest version
path = kagglehub.dataset_download("uditjain13/heart-disease-risk-2026")

print("Path to dataset files:", path)

100%|██████████| 344k/344k [00:00<00:00, 68.5MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/uditjain13/heart-disease-risk-2026/versions/3


# **2.0 Lectura del conjunto de datos**

In [ ]:
# Cargar dataset
file =os.path.join(path, "heart_disease_risk_2026.csv")
df = pd.read_csv(file)

print(f"Dimensiones del dataset: {df.shape}")
df.head()

Dimensiones del dataset: (9000, 27)


,patient_id,age,sex,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,1,44,Male,117,74,193,57,106,119,112,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,2,57,Male,139,94,185,69,110,35,114,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,3,29,Male,128,78,197,52,108,157,95,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0
3,4,72,Male,132,86,197,59,104,143,92,...,False,Never,2.7,18,6.8,63.6,True,7373,48.5,1
4,5,62,Female,116,75,154,65,75,104,135,...,True,Former,3.3,24,8.2,58.7,False,6331,47.3,1


# **3. Exploración básica**

In [ ]:
# Exploración básica

print("Dimensiones:", df.shape)

print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nDuplicados:")
print(df.duplicated().sum())

print("\nPrimeras filas:")
display(df.head())

Dimensiones: (9000, 27)

Columnas:
['patient_id', 'age', 'sex', 'resting_bp_systolic', 'resting_bp_diastolic', 'cholesterol_total', 'hdl', 'ldl', 'triglycerides', 'fasting_blood_sugar', 'hba1c', 'bmi', 'resting_heart_rate', 'max_heart_rate_achieved', 'chest_pain_type', 'exercise_induced_angina', 'st_depression', 'family_history', 'smoker_status', 'alcohol_units_per_week', 'exercise_minutes_per_week', 'sleep_hours', 'stress_score', 'wearable_owner', 'daily_steps', 'diet_quality_score', 'has_heart_disease']

Tipos de datos:
patient_id                     int64
age                            int64
sex                           object
resting_bp_systolic            int64
resting_bp_diastolic           int64
cholesterol_total              int64
hdl                            int64
ldl                            int64
triglycerides                  int64
fasting_blood_sugar            int64
hba1c                        float64
bmi                          float64
resting_heart_rate          

,patient_id,age,sex,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,...,family_history,smoker_status,alcohol_units_per_week,exercise_minutes_per_week,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease
0,1,44,Male,117,74,193,57,106,119,112,...,False,Never,2.9,86,5.4,19.8,True,7731,62.9,0
1,2,57,Male,139,94,185,69,110,35,114,...,False,Never,3.0,132,4.3,45.8,True,2629,74.6,1
2,3,29,Male,128,78,197,52,108,157,95,...,False,Current,3.5,128,5.1,17.7,True,9290,65.7,0
3,4,72,Male,132,86,197,59,104,143,92,...,False,Never,2.7,18,6.8,63.6,True,7373,48.5,1
4,5,62,Female,116,75,154,65,75,104,135,...,True,Former,3.3,24,8.2,58.7,False,6331,47.3,1


# **4. Transformación de datos**

In [ ]:
def transforma_categoricas_nominal(df, categoricas_nominales):

    df_encoded = df.copy()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

    for col in categoricas_nominales:
        encoded_data = encoder.fit_transform(df_encoded[[col]])
        feature_names = encoder.get_feature_names_out([col])
        encoded_df = pd.DataFrame(encoded_data, columns=feature_names, index=df_encoded.index)

        df_encoded = pd.concat([df_encoded.drop(columns=[col]), encoded_df], axis=1)

    return df_encoded


In [ ]:
def transforma_categoricas_ordinal(df, categoricas_ordinales):
    df_encoded = df.copy()

    for col_name, categories in categoricas_ordinales:
        if col_name in df_encoded.columns:
            encoder = OrdinalEncoder(categories=[categories], handle_unknown='use_encoded_value', unknown_value=-1)
            df_encoded[col_name] = encoder.fit_transform(df_encoded[[col_name]])
        else:
            print(f"Warning: Column '{col_name}' not found in DataFrame. Skipping ordinal encoding for this column.")

    return df_encoded


### Aplicación de las funciones de codificación

Primero identificaremos las columnas nominales y las codificaremos. Para la codificación ordinal, ya que no hay columnas inherentemente ordinales claras en este dataset, haremos una demostración con la columna `chest_pain_type` como ejemplo, definiendo un orden arbitrario para fines ilustrativos.

In [ ]:
# Identificar columnas nominales (excluyendo chest_pain_type, que se tratará como ordinal)
nominal_cols = ['sex', 'smoker_status']

# Aplicar la codificación one-hot
df_intermedio = transforma_categoricas_nominal(df, nominal_cols)

print("DataFrame después de la codificación nominal (sex, smoker_status):")
display(df_intermedio.head())


DataFrame después de la codificación nominal (sex, smoker_status):


,patient_id,age,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,...,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease,sex_Female,sex_Male,smoker_status_Current,smoker_status_Former,smoker_status_Never
0,1,44,117,74,193,57,106,119,112,5.2,...,19.8,True,7731,62.9,0,0.0,1.0,0.0,0.0,1.0
1,2,57,139,94,185,69,110,35,114,5.9,...,45.8,True,2629,74.6,1,0.0,1.0,0.0,0.0,1.0
2,3,29,128,78,197,52,108,157,95,5.5,...,17.7,True,9290,65.7,0,0.0,1.0,1.0,0.0,0.0
3,4,72,132,86,197,59,104,143,92,4.8,...,63.6,True,7373,48.5,1,0.0,1.0,0.0,0.0,1.0
4,5,62,116,75,154,65,75,104,135,6.2,...,58.7,False,6331,47.3,1,1.0,0.0,0.0,1.0,0.0


In [ ]:
chest_pain_order = ['Asymptomatic', 'Non-Anginal Pain', 'Atypical Angina', 'Typical Angina']

# Lista de tuplas: (nombre_columna, lista_de_categorias_en_orden)
ordinal_cols_to_encode = [('chest_pain_type', chest_pain_order)]

# Aplicar la codificación ordinal al DataFrame intermedio
data = transforma_categoricas_ordinal(df_intermedio, ordinal_cols_to_encode)

print("\nDataFrame final después de ambas codificaciones (nominal y ordinal):")
display(data.head())



DataFrame final después de ambas codificaciones (nominal y ordinal):


,patient_id,age,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,...,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease,sex_Female,sex_Male,smoker_status_Current,smoker_status_Former,smoker_status_Never
0,1,44,117,74,193,57,106,119,112,5.2,...,19.8,True,7731,62.9,0,0.0,1.0,0.0,0.0,1.0
1,2,57,139,94,185,69,110,35,114,5.9,...,45.8,True,2629,74.6,1,0.0,1.0,0.0,0.0,1.0
2,3,29,128,78,197,52,108,157,95,5.5,...,17.7,True,9290,65.7,0,0.0,1.0,1.0,0.0,0.0
3,4,72,132,86,197,59,104,143,92,4.8,...,63.6,True,7373,48.5,1,0.0,1.0,0.0,0.0,1.0
4,5,62,116,75,154,65,75,104,135,6.2,...,58.7,False,6331,47.3,1,1.0,0.0,0.0,1.0,0.0


In [ ]:
def remove_columns(df, columns_to_remove):
    df_cleaned = df.copy()
    existing_columns = [col for col in columns_to_remove if col in df_cleaned.columns]
    non_existing_columns = [col for col in columns_to_remove if col not in df_cleaned.columns]

    if existing_columns:
        df_cleaned = df_cleaned.drop(columns=existing_columns)
        print(f"Columns {existing_columns} removed successfully.")
    else:
        print("No specified columns found in DataFrame to remove.")

    if non_existing_columns:
        print(f"Warning: Columns {non_existing_columns} not found in DataFrame. Skipping their removal.")

    return df_cleaned


columns_to_drop = ['patient_id']
data = remove_columns(data, columns_to_drop)

print("\nDataFrame final después de eliminar las columnas especificadas:")
display(data.head())
print(f"Dimensiones del DataFrame final: {data.shape}")

No specified columns found in DataFrame to remove.

DataFrame final después de eliminar las columnas especificadas:


,age,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,bmi,...,stress_score,wearable_owner,daily_steps,diet_quality_score,has_heart_disease,sex_Female,sex_Male,smoker_status_Current,smoker_status_Former,smoker_status_Never
0,44,117,74,193,57,106,119,112,5.2,26.5,...,19.8,True,7731,62.9,0,0.0,1.0,0.0,0.0,1.0
1,57,139,94,185,69,110,35,114,5.9,20.8,...,45.8,True,2629,74.6,1,0.0,1.0,0.0,0.0,1.0
2,29,128,78,197,52,108,157,95,5.5,24.5,...,17.7,True,9290,65.7,0,0.0,1.0,1.0,0.0,0.0
3,72,132,86,197,59,104,143,92,4.8,27.3,...,63.6,True,7373,48.5,1,0.0,1.0,0.0,0.0,1.0
4,62,116,75,154,65,75,104,135,6.2,26.0,...,58.7,False,6331,47.3,1,1.0,0.0,0.0,1.0,0.0


Dimensiones del DataFrame final: (9000, 29)


In [ ]:
def split_data_train_val_test(data, target_column, test_size=0.15, val_size=0.15, random_state=42):

    X = data.drop(columns=[target_column])
    y = data[target_column]

    # First split: Separate out the test set
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    # Calculate the proportion for the validation set from the remaining training-validation set
    # (val_size / (1 - test_size))
    val_size_adjusted = val_size / (1 - test_size)

    # Second split: Separate out the validation set from the remaining training-validation set
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_size_adjusted, random_state=random_state, stratify=y_train_val
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# Definir la columna objetivo
target = 'has_heart_disease'

# Aplicar la función de división al DataFrame escalado
X_train, X_val, X_test, y_train, y_val, y_test = split_data_train_val_test(data, target)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_val: {X_val.shape}")
print(f"Dimensiones de X_test: {X_test.shape}")
print(f"Dimensiones de y_train: {y_train.shape}")
print(f"Dimensiones de y_val: {y_val.shape}")
print(f"Dimensiones de y_test: {y_test.shape}")

Dimensiones de X_train: (6300, 28)
Dimensiones de X_val: (1350, 28)
Dimensiones de X_test: (1350, 28)
Dimensiones de y_train: (6300,)
Dimensiones de y_val: (1350,)
Dimensiones de y_test: (1350,)


In [ ]:
def check_ordinal_categories(df, ordinal_cols_with_categories):
    print("\n--- Verificación de Columnas Ordinales ---")
    for col_name, categories in ordinal_cols_with_categories:
        if col_name in df.columns:
            unique_vals_in_df = df[col_name].unique()
            num_unique_in_df = len(unique_vals_in_df)
            num_categories_provided = len(categories)

            print(f"\nColumna '{col_name}':")
            print(f"  Número de categorías únicas en el DataFrame: {num_unique_in_df}")
            print(f"  Número de categorías en la lista proporcionada: {num_categories_provided}")

            if num_unique_in_df > 1:
                print(f"La columna '{col_name}' tiene más de una categoría.\n")
            else:
                print(f"La columna '{col_name}' no tiene más de una categoría.\n")

            # Optionally, check if all unique values in the column are present in the provided categories
            missing_from_categories = [val for val in unique_vals_in_df if val not in categories]
            if missing_from_categories:
                print(f"Valores en el DataFrame no encontrados en la lista de categorías proporcionada: {missing_from_categories}\n")

            missing_from_df = [val for val in categories if val not in unique_vals_in_df]
            if missing_from_df:
                print(f"Valores en la lista de categorías proporcionada no encontrados en el DataFrame: {missing_from_df}\n")

        else:
            print(f"\nColumna '{col_name}':No encontrada en el DataFrame. Saltando verificación.\n")

# Demostración de la función con el DataFrame 'data' y las columnas ordinales ya definidas
check_ordinal_categories(data, ordinal_cols_to_encode)



--- Verificación de Columnas Ordinales ---

Columna 'chest_pain_type':
  Número de categorías únicas en el DataFrame: 4
  Número de categorías en la lista proporcionada: 4
La columna 'chest_pain_type' tiene más de una categoría.

Valores en el DataFrame no encontrados en la lista de categorías proporcionada: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)]

Valores en la lista de categorías proporcionada no encontrados en el DataFrame: ['Asymptomatic', 'Non-Anginal Pain', 'Atypical Angina', 'Typical Angina']



In [ ]:
from sklearn.preprocessing import StandardScaler

def scale_numerical_features(df_to_scale, exclude_cols, scaler=None):
    df_scaled = df_to_scale.copy()

    # Identify numerical columns to scale
    numerical_cols_to_scale = [
        col for col in df_scaled.select_dtypes(include=np.number).columns
        if col not in exclude_cols
    ]

    if not numerical_cols_to_scale:
        print("No numerical columns found to scale after exclusions.")
        return df_scaled, scaler # Return original df and scaler

    if scaler is None:
        scaler = StandardScaler()
        df_scaled[numerical_cols_to_scale] = scaler.fit_transform(df_scaled[numerical_cols_to_scale])
        print(f"New StandardScaler fitted and columns scaled: {numerical_cols_to_scale}")
    else:
        # If a scaler is provided, assume it's already fitted and just transform
        df_scaled[numerical_cols_to_scale] = scaler.transform(df_scaled[numerical_cols_to_scale])
        print(f"Existing StandardScaler used to transform columns: {numerical_cols_to_scale}")

    return df_scaled, scaler

excluir_del_escalado = [
    'sex_Male', 'sex_Female',
    'smoker_status_Current', 'smoker_status_Former', 'smoker_status_Never',
    'chest_pain_type', # Codificación ordinal
    'exercise_induced_angina', 'family_history', 'wearable_owner', # Boolean/Binary
    'has_heart_disease' # Variable objetivo
]

# Escalar el conjunto de entrenamiento (fit_transform)

X_train_scaled, fitted_scaler = scale_numerical_features(X_train, excluir_del_escalado, scaler=None)

# Escalar los conjuntos de validación y prueba (solo transform)

X_val_scaled, _ =  scale_numerical_features(X_val, excluir_del_escalado, scaler=fitted_scaler)
X_test_scaled, _ = scale_numerical_features(X_test, excluir_del_escalado, scaler=fitted_scaler)

print("\nPrimeras filas de X_train_scaled:")
display(X_train_scaled.head())
print("\nPrimeras filas de X_val_scaled:")
display(X_val_scaled.head())
print("\nPrimeras filas de X_test_scaled:")
display(X_test_scaled.head())

New StandardScaler fitted and columns scaled: ['age', 'resting_bp_systolic', 'resting_bp_diastolic', 'cholesterol_total', 'hdl', 'ldl', 'triglycerides', 'fasting_blood_sugar', 'hba1c', 'bmi', 'resting_heart_rate', 'max_heart_rate_achieved', 'st_depression', 'alcohol_units_per_week', 'exercise_minutes_per_week', 'sleep_hours', 'stress_score', 'daily_steps', 'diet_quality_score']
Existing StandardScaler used to transform columns: ['age', 'resting_bp_systolic', 'resting_bp_diastolic', 'cholesterol_total', 'hdl', 'ldl', 'triglycerides', 'fasting_blood_sugar', 'hba1c', 'bmi', 'resting_heart_rate', 'max_heart_rate_achieved', 'st_depression', 'alcohol_units_per_week', 'exercise_minutes_per_week', 'sleep_hours', 'stress_score', 'daily_steps', 'diet_quality_score']
Existing StandardScaler used to transform columns: ['age', 'resting_bp_systolic', 'resting_bp_diastolic', 'cholesterol_total', 'hdl', 'ldl', 'triglycerides', 'fasting_blood_sugar', 'hba1c', 'bmi', 'resting_heart_rate', 'max_heart_rat

,age,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,bmi,...,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,sex_Female,sex_Male,smoker_status_Current,smoker_status_Former,smoker_status_Never
2683,-0.695349,-1.363484,-1.168610,-1.176392,-1.314927,-0.676652,-1.191761,2.192045,1.617229,1.411062,...,0.187629,-1.822959,True,0.691405,0.879874,0.0,1.0,0.0,1.0,0.0
1380,-0.849896,0.228467,-0.072658,-0.093027,-1.719117,0.421017,-0.121323,-0.054489,-0.989388,-1.077943,...,0.734415,-1.520442,False,0.660097,-1.031342,0.0,1.0,1.0,0.0,0.0
190,0.077384,-1.074038,-1.807915,-0.804952,-0.425709,-0.932774,0.287390,-0.671184,-0.120516,-0.778341,...,-0.541420,1.461505,True,-0.612935,-0.598224,0.0,1.0,0.0,1.0,0.0
4381,1.545576,-0.350424,-0.072658,0.773665,1.271891,0.603962,-1.094449,-0.627135,-0.989388,-1.377546,...,-0.450289,0.492218,False,0.041307,0.714877,1.0,0.0,0.0,0.0,1.0
332,0.386477,0.228467,1.114624,-1.021625,-0.506547,-1.115719,-0.238098,0.430058,0.748356,-0.086951,...,-0.541420,0.955253,True,-0.048013,-0.811346,1.0,0.0,0.0,0.0,1.0



Primeras filas de X_val_scaled:


,age,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,bmi,...,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,sex_Female,sex_Male,smoker_status_Current,smoker_status_Former,smoker_status_Never
237,0.772843,1.096804,1.388612,-0.866859,0.867700,-1.445020,-0.277023,-0.230688,0.458732,-0.156090,...,-0.905944,2.109755,False,-0.156669,-0.020735,1.0,0.0,1.0,0.0,0.0
5916,-0.463529,-0.422786,-1.625256,0.030786,0.544348,0.164894,-1.600474,-1.508128,-1.279012,-1.930658,...,1.827988,1.794891,False,-0.734023,-0.536350,1.0,0.0,0.0,0.0,1.0
2572,0.541023,1.024442,0.657977,1.114151,0.382672,1.116207,-0.335410,0.958654,0.893168,0.373976,...,0.461022,-1.335228,False,-1.396552,0.350508,1.0,0.0,0.0,0.0,1.0
3521,0.927390,0.373190,1.114624,0.185553,-1.072413,0.384428,1.299442,-0.583085,-1.279012,1.871989,...,0.278760,-0.822803,False,-1.293421,-0.715097,0.0,1.0,0.0,0.0,1.0
6042,-1.931722,-1.146400,-0.894622,-0.928765,-0.910737,-1.115719,0.287390,0.606257,0.458732,-0.086951,...,0.005367,0.455175,False,-0.677853,-0.185732,0.0,1.0,0.0,0.0,1.0



Primeras filas de X_test_scaled:


,age,resting_bp_systolic,resting_bp_diastolic,cholesterol_total,hdl,ldl,triglycerides,fasting_blood_sugar,hba1c,bmi,...,sleep_hours,stress_score,wearable_owner,daily_steps,diet_quality_score,sex_Female,sex_Male,smoker_status_Current,smoker_status_Former,smoker_status_Never
6884,2.009216,0.879719,0.201330,-1.207345,-0.587385,-1.335253,-0.821973,0.121710,-0.554952,-1.147082,...,-0.905944,1.745500,True,0.425749,-0.632599,0.0,1.0,0.0,0.0,1.0
8735,1.081937,0.300828,0.931965,-0.526373,1.271891,-1.005952,-0.685736,1.531300,1.472417,-0.593970,...,0.643284,1.158989,True,-0.204552,0.701127,1.0,0.0,0.0,1.0,0.0
1430,-0.386256,0.952081,0.110001,1.485590,0.382672,1.408919,1.026966,0.606257,0.458732,-0.617016,...,-0.176895,1.541765,True,0.920689,0.625503,1.0,0.0,0.0,0.0,1.0
2382,-0.154436,-0.133340,1.114624,0.464132,-0.506547,0.384428,0.579328,1.046753,1.037980,0.304837,...,0.096498,-0.291855,True,1.249882,0.549880,1.0,0.0,0.0,0.0,1.0
8019,0.463750,0.083744,-0.163987,0.123646,1.029376,-0.603474,0.170615,1.046753,0.313920,0.097420,...,-0.997075,1.603503,False,-0.451792,1.257992,1.0,0.0,0.0,0.0,1.0
